# Tutorial: Understanding GAMMA's Adaptive Multi-Hop Routing

**Audience:** TopoBench users who want to understand or inspect GAMMA's routing mechanism.

**Prerequisites:** Basic PyTorch tensors, PyG's `edge_index` convention, and a working TopoBench environment.

**Learning goals:** By the end, you can explain why one routing pass is uniform, inspect the node-specific hop weights used by GAMMA, and compare routing on small homophilic and heterophilic graphs.

The implementation follows Algorithm 1 of [GAMMA (NeurIPS 2025)](https://proceedings.neurips.cc/paper_files/paper/2025/file/1129f729097a28bfb5b836ec4bf94478-Paper-Conference.pdf). This tutorial illustrates the mechanism; it is not a benchmark result.


## Outline

1. Build a deterministic three-node example.
2. Verify that one routing pass is uniform.
3. Inspect the adaptive weights from two routing passes.
4. Compare two graph regimes.
5. Try a third routing pass and review common pitfalls.


In [ ]:
from __future__ import annotations

import torch

from topobench.nn.backbones.graph.gamma import GAMMA

torch.manual_seed(7)
torch.set_printoptions(precision=4, sci_mode=False)


def make_gamma(routing_iterations: int) -> GAMMA:
    """Create a deterministic paper-faithful GAMMA layer."""
    model = GAMMA(
        in_channels=2,
        out_channels=2,
        max_hops=2,
        num_routing_iterations=routing_iterations,
        bias=False,
    ).double()
    with torch.no_grad():
        model.projection.weight.copy_(torch.eye(2, dtype=torch.float64))
        model.hop_scale.fill_(1.0)
    return model


## 1. A deterministic path graph

We use the undirected path `0 -- 1 -- 2`, identity projection, and unit channel gates. This isolates routing from learned projection effects. PyG stores every undirected edge in both directions.


In [ ]:
x = torch.tensor(
    [[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0]],
    dtype=torch.float64,
)
edge_index = torch.tensor(
    [[0, 1, 1, 2], [1, 0, 2, 1]],
    dtype=torch.long,
)

x, edge_index


## 2. Why one pass is not adaptive

Routing logits start at zero. Their first softmax is therefore exactly uniform. The agreement update occurs *after* the first output is formed, so `R=1` cannot use it.


In [ ]:
one_pass = make_gamma(routing_iterations=1)
one_pass_weights = one_pass.get_routing_weights(x, edge_index)
expected_uniform = torch.full_like(one_pass_weights, 1.0 / 3.0)
torch.testing.assert_close(one_pass_weights, expected_uniform)

one_pass_weights


## 3. Two passes produce node-specific hop weights

With `R=2`, the second softmax uses one agreement update. Each row below corresponds to a node; columns are the 0-, 1-, and 2-hop candidates. Every row remains a probability simplex.


In [ ]:
two_pass = make_gamma(routing_iterations=2)
two_pass_weights = two_pass.get_routing_weights(x, edge_index)
two_pass_output = two_pass(x, edge_index)

torch.testing.assert_close(
    two_pass_weights.sum(dim=1),
    torch.ones(3, dtype=torch.float64),
)
print("routing weights [node, hop]:\n", two_pass_weights)
print("node embeddings:\n", two_pass_output)


The middle node keeps uniform weights because all three normalized candidates point in the same feature direction. The end nodes distinguish their candidates and favor hop 1 slightly. These are the coefficients that actually formed the returned embedding—not a softmax after an unused final update.


## 4. A small graph-regime comparison

Next, identical-feature nodes are either connected to each other (homophilic edges) or to opposite-feature nodes (heterophilic edges). The layer is still untrained, so this is a controlled mechanism check rather than evidence of predictive performance.


In [ ]:
regime_x = torch.tensor(
    [[1.0, 0.0], [1.0, 0.0], [-1.0, 0.0], [-1.0, 0.0]],
    dtype=torch.float64,
)
homophilic_edges = torch.tensor(
    [[0, 1, 2, 3], [1, 0, 3, 2]],
    dtype=torch.long,
)
heterophilic_edges = torch.tensor(
    [[0, 2, 1, 3], [2, 0, 3, 1]],
    dtype=torch.long,
)

homophilic_mean = two_pass.get_routing_weights(
    regime_x, homophilic_edges
).mean(dim=0)
heterophilic_mean = two_pass.get_routing_weights(
    regime_x, heterophilic_edges
).mean(dim=0)

print("mean weights [0-hop, 1-hop, 2-hop]")
print("homophilic:  ", homophilic_mean)
print("heterophilic:", heterophilic_mean)


## Exercise: add one more routing pass

Before running the answer, predict whether `R=3` will sharpen or flatten the preferences seen for `R=2`. The answer cell constructs the same layer with one extra agreement update.


In [ ]:
three_pass = make_gamma(routing_iterations=3)
three_pass_weights = three_pass.get_routing_weights(x, edge_index)

print("R=2:\n", two_pass_weights)
print("R=3:\n", three_pass_weights)
assert not torch.allclose(three_pass_weights, two_pass_weights)


## Pitfalls and extensions

- **Common mistake:** calling `R=1` adaptive. Use at least two passes.
- **Propagation semantics:** the paper-faithful default adds remaining self-loops and applies symmetric normalization. Use `propagation="raw"` only for authors-code compatibility. No dense adjacency power is created.
- **Interpretation:** routing weights explain hop selection, not class importance or causal influence.
- **Extension:** after official GraphUniverse training, aggregate `get_routing_weights` by homophily, degree, and power-law regime. Report model commit, seed, and graph counts so the comparison is reproducible.

You now have a deterministic mechanism test and the inspection API needed for a larger structural analysis.
